# AVA T0와 K4-1000 · LIBERO+ Long paired 평가

Colab에서 **A100 GPU**를 선택하고 1~5번 셀을 순서대로 실행하세요. 6번 셀은 저장된 결과를 다시 읽어 비교할 때 실행하세요. 4번까지는 모델 rollout을 실행하지 않습니다. 5번을 실행할 때마다 별도 Drive 폴더에 T0 평가 15회를 기록합니다.

Colab Secrets의 `HF_TOKEN`에 비공개 T0 체크포인트 읽기 권한이 필요합니다. GitHub 저장소 접근에 인증이 필요한 경우 `GH_TOKEN`도 설정하세요. 체크포인트가 다른 검증된 캐시 경로에 있으면 1번 셀의 `T0_CHECKPOINT`를 수정하세요.

기준은 [K4-1000 episode-only 실행](https://drive.google.com/drive/folders/1ySB5vkS5vRNhWQRGa7Tc5hCwkPEnr86_)입니다. 체크포인트 위치와 정규화 키 선택은 [이전 T0 평가](https://colab.research.google.com/drive/1DY8UkgDcY86cQLE3Wwt_OutHyaPGoBC6)를 따릅니다. [Temporal context 평가](https://chatgpt.com/library?file_id=libfile_1984a9282dec81918dab287bbd0af284&account_id=personal)의 초기 상태 해시와 episode-only 추적을 재사용합니다.

In [ ]:
# @title 1. Drive 마운트 및 평가 경로
from pathlib import Path, PurePosixPath
from google.colab import drive, userdata
import base64
import datetime
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import zipfile

K4_DIR = Path('/content/drive/MyDrive/VLADE/evaluations/libero_plus_ava_temporal_context/20261112T094936298118Z/episode_only')
OUTPUT_ROOT = Path('/content/drive/MyDrive/VLADE/evaluations/libero_plus_ava_t0_k4_long_paired')
T0_CHECKPOINT = 'hfasset://ava_t0'  # 또는 등록된 ava_t0의 검증된 로컬 캐시 경로
T0_UNNORM_KEY = ''  # 자동: 이전 평가와 같이 libero_10, 다음 libero_10_no_noops
VLADE_COMMIT = 'e9d30b7c4278670b6d580e991eafcd885c68ca51'
AVA_COMMIT = '69ae5dfb90f4eb714259693bd2c222e0b3de6456'
WORK = Path('/content/vlade_t0_k4_long_paired')
MUJOCO_VERSION = '3.3.1'
drive.mount('/content/drive')

In [ ]:
# @title 2. K4 설정·태스크 순서·영상 확인과 비교 함수
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path, value):
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)


def argument_map(argv):
    keys, values = argv[::2], argv[1::2]
    if len(keys) != len(values) or len(set(keys)) != len(keys) or not all(
        isinstance(key, str) and key.startswith('--') for key in keys
    ):
        raise ValueError('run.json의 평가 인수 형식이 올바르지 않습니다.')
    return dict(zip(keys, values))


def differences(expected, actual):
    return {
        key: {'expected': expected.get(key), 'actual': actual.get(key)}
        for key in sorted(expected.keys() | actual.keys())
        if key not in expected or key not in actual or expected[key] != actual[key]
    }


def require_match(label, expected, actual):
    mismatch = differences(expected, actual)
    print(label, json.dumps(mismatch, ensure_ascii=False, indent=2) if mismatch else '일치')
    if mismatch:
        raise ValueError(f'{label} 불일치')


def by_task(rows):
    result = {}
    for row in rows:
        task_id = row['task_id']
        if type(task_id) is not int or task_id in result:
            raise ValueError(f'태스크 ID가 잘못되었거나 중복입니다. {task_id!r}')
        result[task_id] = row
    return result


def compare_results(k4_rows, t0_rows, k4_videos):
    k4_by_id, t0_by_id = by_task(k4_rows), by_task(t0_rows)
    names = {(True, True): 'both_success', (True, False): 't0_only',
             (False, True): 'k4_only', (False, False): 'both_failure'}
    groups = {name: {'count': 0, 'task_ids': []} for name in names.values()}
    paired, unmatched, t0_only_videos = [], [], []
    for task_id in dict.fromkeys([*k4_by_id, *t0_by_id]):
        k4, t0 = k4_by_id.get(task_id), t0_by_id.get(task_id)
        reason = None
        if k4 is None or t0 is None:
            reason = 'missing_k4_task' if k4 is None else 'missing_t0_task'
        else:
            k4_hash = k4.get('initial_state_sha256')
            t0_hash = t0.get('initial_state_sha256')
            if not all(isinstance(h, str) and re.fullmatch(r'[0-9a-f]{64}', h)
                       for h in (k4_hash, t0_hash)):
                reason = 'missing_or_invalid_initial_state_sha256'
            elif k4_hash != t0_hash:
                reason = 'initial_state_sha256_mismatch'
            elif type(k4.get('success')) is not bool or type(t0.get('success')) is not bool:
                reason = 'missing_or_invalid_success'
        if reason:
            unmatched.append({
                'task_id': task_id, 'reason': reason,
                'k4_initial_state_sha256': (k4 or {}).get('initial_state_sha256'),
                't0_initial_state_sha256': (t0 or {}).get('initial_state_sha256'),
            })
            continue

        group = names[t0['success'], k4['success']]
        groups[group]['task_ids'].append(task_id)
        groups[group]['count'] += 1
        paired.append({'task_id': task_id, 'initial_state_sha256': t0_hash,
                       't0_success': t0['success'], 'k4_success': k4['success']})
        if group == 't0_only':
            t0_only_videos.append({
                'task_id': task_id, 't0_video_path': t0.get('video_path'),
                'k4_video_path': k4_videos.get(task_id),
            })
    scored = [row for row in t0_rows if type(row.get('success')) is bool]
    successes = sum(row['success'] for row in scored)
    return {
        'groups': groups, 'paired_count': len(paired), 'paired_results': paired,
        'unmatched': unmatched, 't0_only_videos': t0_only_videos,
        't0': {'successes': successes, 'rollouts': len(scored),
               'success_rate': successes / len(scored) if scored else None},
    }


def k4_video_paths(folder, rows):
    # K4 평가기가 저장한 전역 episode 번호는 task_results.json의 1-based 순서입니다.
    indexed = {}
    for path in (folder / 'rollouts').rglob('*.mp4'):
        match = re.search(r'--episode=(\d+)--success=(True|False)--task=', path.name)
        if match:
            indexed.setdefault(int(match[1]), []).append((match[2], path))
    paths = {}
    for episode, row in enumerate(rows, 1):
        matches = indexed.get(episode, [])
        if len(matches) == 1 and matches[0][0] == str(row['success']):
            paths[row['task_id']] = str(matches[0][1])
        else:
            print(f"K4 영상 확인 필요 · 태스크 {row['task_id']} · episode {episode} · {matches}")
    return paths


K4_RUN = read_json(K4_DIR / 'run.json')
K4_ROWS = read_json(K4_DIR / 'task_results.json')
TASK_IDS = list(by_task(K4_ROWS))
if len(TASK_IDS) != 15 or any(type(row.get('success')) is not bool for row in K4_ROWS):
    raise ValueError('K4 Long 결과에 완료된 rollout 15개가 필요합니다.')

require_match('K4 기준 실행', {
    'condition': 'episode_only', 'reset_every_policy_queries': None,
    'task_ids': TASK_IDS, 'successes': 7,
}, {
    'condition': K4_RUN['condition'],
    'reset_every_policy_queries': K4_RUN['reset_every_policy_queries'],
    'task_ids': K4_RUN['task_ids'], 'successes': sum(row['success'] for row in K4_ROWS),
})
if any(row.get('context_reset_queries') != [1] for row in K4_ROWS):
    raise ValueError('K4 태스크의 context reset 기록이 episode-only와 다릅니다.')

K4_ARGS = argument_map(K4_RUN['args'])
RUN_PATH_ARGS = {'--pretrained_checkpoint', '--local_log_dir'}
COMMON_ARGS = {key: value for key, value in K4_ARGS.items() if key not in RUN_PATH_ARGS}
# 2026-11-12 K4 run.json에서 확인한 조건. 실행 인수는 위에서 읽은 값을 사용합니다.
EXPECTED_ARGS = {
    "--model_family": "openvla",
    "--use_l1_regression": "True",
    "--use_diffusion": "False",
    "--use_proprio": "True",
    "--num_images_in_input": "2",
    "--num_open_loop_steps": "8",
    "--task_suite_name": "libero_10",
    "--num_trials_per_task": "1",
    "--seed": "42",
    "--use_wandb": "False",
    "--multi_frame.temporal_strategy": "attn_weight",
    "--multi_frame.temporal_feature_source": "action",
    "--multi_frame.temporal_feature_layer": "-2",
    "--multi_frame.attn_weight_force_eager_attn": "True",
    "--multi_frame.attn_weight_extra_layer_ids": "(i for i in range(0, 32))",
    "--multi_frame.attn_weight_head_type": "softmaxscore",
    "--multi_frame.attn_weight_score_config": "(1.9, 0.1, 0.0)",
    "--multi_frame.attn_weight_sink_ids": "[68,75,180,187,324,331,436,443]",
    "--multi_frame.attn_weight_sink_weight": "1.0"
}
require_match('K4 CLI 설정', EXPECTED_ARGS,
              {k: v for k, v in COMMON_ARGS.items() if k != '--initial_states_path'})
require_match('K4 초기 상태', {'initial_states_path': 'DEFAULT'},
              {'initial_states_path': COMMON_ARGS.get('--initial_states_path', 'DEFAULT')})
SUITE = COMMON_ARGS['--task_suite_name']
LIBERO_REV = K4_RUN['libero_plus_revision']
ASSETS_REV = K4_RUN['assets_revision']
K4_VIDEOS = k4_video_paths(K4_DIR, K4_ROWS)
print('K4 task_results.json 순서', TASK_IDS)
print('공통 CLI 설정', json.dumps(COMMON_ARGS, indent=2))

def save_comparison(folder):
    t0_run = read_json(folder / 'run.json')
    t0_rows = read_json(folder / 'task_results.json')
    summary = compare_results(K4_ROWS, t0_rows, K4_VIDEOS)
    summary.update({
        'suite': t0_run['suite'], 'planned_task_ids': TASK_IDS,
        'k4_run_dir': str(K4_DIR), 't0_run_dir': str(folder),
        't0_run_status': t0_run['status'],
        'k4_successes': sum(row['success'] for row in K4_ROWS),
        'k4_rollouts': len(K4_ROWS),
        'shared_setting_differences': t0_run['shared_setting_differences'],
        'model_setting_differences': t0_run['model_setting_differences'],
    })
    write_json(folder / 'comparison.json', summary)
    labels = {'both_success': '둘 다 성공', 't0_only': 'T0만 성공',
              'k4_only': 'K4만 성공', 'both_failure': '둘 다 실패'}
    for key, label in labels.items():
        group = summary['groups'][key]
        print(f"{label} {group['count']}개 · {group['task_ids']}")
    for row in summary['t0_only_videos']:
        print(f"태스크 {row['task_id']}\n  T0: {row['t0_video_path']}\n  K4: {row['k4_video_path']}")
    for row in summary['unmatched']:
        print('paired 집계 제외', json.dumps(row, ensure_ascii=False))
    t0 = summary['t0']
    rate = f"{t0['success_rate']:.1%}" if t0['success_rate'] is not None else '미집계'
    print(f"이번 T0 성공률 {t0['successes']}/{t0['rollouts']} ({rate})")
    print('결과', folder / 'comparison.json')
    return summary

In [ ]:
# @title 3. 고정 소스·평가 환경·LIBERO+ assets 준비
WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK / VLADE_COMMIT / 'VLADE'
AVA = PROJECT / 'third_party/ava_vla'
SCRIPT = AVA / 'experiments/robot/libero/run_libero_eval.py'
LIBERO = WORK / f'LIBERO-plus-{LIBERO_REV}-{ASSETS_REV}'
VENV = WORK / f'py311-mujoco{MUJOCO_VERSION}-{VLADE_COMMIT}'
PYTHON = VENV / 'bin/python'


def cmd(*args, **kwargs):
    subprocess.run([str(arg) for arg in args], check=True, **kwargs)


def capture(*args, **kwargs):
    return subprocess.check_output([str(arg) for arg in args], text=True, **kwargs).strip()


def optional_secret(name):
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None


def github_environment():
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = optional_secret('GH_TOKEN')
    if token:
        auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        index = int(env.get('GIT_CONFIG_COUNT', '0'))
        env.update({'GIT_CONFIG_COUNT': str(index + 1),
                    f'GIT_CONFIG_KEY_{index}': 'http.https://github.com/.extraheader',
                    f'GIT_CONFIG_VALUE_{index}': f'Authorization: Basic {auth}'})
    return env


def pinned_checkout(url, revision, destination):
    if not re.fullmatch(r'[0-9a-f]{40}', revision):
        raise ValueError(f'고정 Git commit이 필요합니다. {revision!r}')
    if not (destination / '.git').exists():
        destination.mkdir(parents=True, exist_ok=True)
        cmd('git', 'init', '-q', destination)
        cmd('git', '-C', destination, 'remote', 'add', 'origin', url)
    head = subprocess.run(['git', '-C', str(destination), 'rev-parse', '--verify', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()
    if not head:
        cmd('git', '-C', destination, 'fetch', '--depth=1', 'origin', revision,
            env=github_environment())
        cmd('git', '-C', destination, 'checkout', '--detach', 'FETCH_HEAD')
    return capture('git', '-C', destination, 'rev-parse', 'HEAD')


def verify_evaluator_source(source, recorded_sha):
    raw_sha = hashlib.sha256(source).hexdigest()
    if raw_sha == recorded_sha:
        return {'sha256': raw_sha, 'match': 'exact'}

    lf = source.replace(b'\r\n', b'\n')
    # A Windows checkout and the Colab checkout can use different line endings.
    alternate_sha = None
    if b'\r' not in lf and b'\n' in lf:
        crlf = lf.replace(b'\n', b'\r\n')
        if source == lf or source == crlf:
            alternate = crlf if source == lf else lf
            alternate_sha = hashlib.sha256(alternate).hexdigest()
    if alternate_sha == recorded_sha and alternate_sha is not None:
        return {'sha256': raw_sha, 'match': 'lf_crlf'}
    raise ValueError(
        f'Evaluator SHA-256 mismatch: K4={recorded_sha}, '
        f'T0={raw_sha}, alternate line endings={alternate_sha}'
    )


ACTUAL_PROJECT = pinned_checkout('https://github.com/eastha10/VLADE.git', VLADE_COMMIT, PROJECT)
require_match('VLADE Git commit', {'commit': VLADE_COMMIT}, {'commit': ACTUAL_PROJECT})
if not (AVA / '.git').exists():
    cmd('git', '-C', PROJECT, 'submodule', 'update', '--init', '--depth=1', '--', 'third_party/ava_vla',
        env=github_environment())
ACTUAL_AVA = capture('git', '-C', AVA, 'rev-parse', 'HEAD')
ACTUAL_LIBERO = pinned_checkout('https://github.com/sylvestf/LIBERO-plus.git', LIBERO_REV, LIBERO)
require_match('평가 소스 버전', {
    'ava_commit': AVA_COMMIT, 'libero_plus_revision': LIBERO_REV,
}, {
    'ava_commit': ACTUAL_AVA, 'libero_plus_revision': ACTUAL_LIBERO,
})
if capture('git', '-C', AVA, 'status', '--porcelain', '--untracked-files=no'):
    raise RuntimeError(f'AVA 소스에 Git 변경이 있습니다. {AVA}')

EVALUATOR_IDENTITY = verify_evaluator_source(SCRIPT.read_bytes(), K4_RUN['evaluator_sha256'])
print('평가기 SHA-256 검증', 'LF/CRLF 변환 후 일치' if EVALUATOR_IDENTITY['match'] == 'lf_crlf' else '일치')

ENV = {
    **os.environ, 'MPLBACKEND': 'Agg', 'MUJOCO_GL': 'egl', 'PYOPENGL_PLATFORM': 'egl',
    'PYTHONPATH': f'{AVA}:{PROJECT / "src"}:{LIBERO}',
    'TOKENIZERS_PARALLELISM': 'false', 'WANDB_MODE': 'disabled',
    'HF_HOME': str(WORK / 'hf'), 'VLADE_ASSET_CACHE': str(WORK / 'verified_hf'),
}
ready = VENV / f'.vlade-ready-{LIBERO_REV}-{ASSETS_REV}'
if not (ready.is_file() and PYTHON.is_file()):
    cmd('apt-get', 'update', '-qq')
    cmd('apt-get', 'install', '-y', 'python3.11', 'python3.11-venv', 'python3.11-dev',
        'build-essential', 'linux-libc-dev', 'libmagickwand-dev', 'ffmpeg')
    if not PYTHON.is_file():
        cmd('python3.11', '-m', 'venv', VENV)
    cmd(PYTHON, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel')
    cmd(PYTHON, '-m', 'pip', 'install', 'torch==2.2.0', 'torchvision==0.17.0',
        'torchaudio==2.2.0', '--index-url', 'https://download.pytorch.org/whl/cu121')
    cmd(PYTHON, '-m', 'pip', 'install', '-e', AVA)
    cmd(PYTHON, '-m', 'pip', 'install', 'evdev', f'mujoco=={MUJOCO_VERSION}', 'robosuite==1.4.1',
        'bddl', 'future', 'easydict==1.9', 'gym==0.25.2', 'termcolor', 'h5py', 'tqdm',
        'PyYAML', 'imageio', 'imageio-ffmpeg==0.6.0', 'Wand==0.7.2', 'scikit-image==0.25.2',
        'numpy==1.26.4', 'opencv-python<4.12', 'tensorflow-metadata==1.15.0')
    cmd(PYTHON, '-m', 'pip', 'install', '--no-deps', '-e', LIBERO)
    ready.touch()


def install_assets(archive, target):
    required = {'textures', 'scenes', 'new_objects'}
    with zipfile.ZipFile(archive) as files:
        entries = []
        found = {}
        for info in files.infolist():
            path = PurePosixPath(info.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in info.filename:
                raise ValueError(f'잘못된 assets ZIP 경로: {info.filename}')
            if info.is_dir() or not path.parts or path.parts[0] == '__MACOSX':
                continue
            entries.append((info, path))
            for index, part in enumerate(path.parts[:-1]):
                if part in required:
                    found.setdefault(path.parts[:index], set()).add(part)
        prefixes = [prefix for prefix, names in found.items() if names == required]
        if len(prefixes) != 1:
            raise ValueError(f'assets ZIP의 중첩 경로를 특정할 수 없습니다. {prefixes}')
        prefix = prefixes[0]
        for info, path in entries:
            if path.parts[:len(prefix)] != prefix:
                continue
            destination = target.joinpath(*path.parts[len(prefix):])
            destination.parent.mkdir(parents=True, exist_ok=True)
            with files.open(info) as source, destination.open('wb') as sink:
                shutil.copyfileobj(source, sink)


assets_root = LIBERO / 'libero/libero'
assets_target = assets_root / 'assets'
assets_receipt = assets_root / '.vlade_assets_revision'
assets_required = ('textures', 'scenes', 'new_objects')
assets_ready = lambda: all((assets_target / name).is_dir() for name in assets_required)
if not (assets_receipt.is_file() and assets_receipt.read_text().strip() == ASSETS_REV and assets_ready()):
    if not re.fullmatch(r'[0-9a-f]{40}', ASSETS_REV):
        raise ValueError(f'고정 HF assets revision이 필요합니다. {ASSETS_REV!r}')
    archive = Path(capture(PYTHON, '-c',
        'from huggingface_hub import hf_hub_download; import sys; '
        'print(hf_hub_download("Sylvest/LIBERO-plus", "assets.zip", '
        'repo_type="dataset", revision=sys.argv[1]))', ASSETS_REV, env=ENV))
    install_assets(archive, assets_target)
    if not assets_ready():
        raise RuntimeError(f'LIBERO+ assets 경로가 올바르지 않습니다. {assets_target}')
    assets_receipt.write_text(ASSETS_REV, encoding='utf-8')
ACTUAL_ASSETS = assets_receipt.read_text(encoding='utf-8').strip()
require_match('LIBERO+ assets', {'revision': ASSETS_REV}, {'revision': ACTUAL_ASSETS})

In [ ]:
# @title 4. A100·MuJoCo 3.3.1·MP4 검사 및 T0 체크포인트 준비
PREFLIGHT = r'''
import importlib.metadata as metadata
import json
import sys
import tempfile
from pathlib import Path

import imageio.v2 as imageio
import mujoco
import numpy as np
import robosuite
import torch
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv
from skimage.filters import gaussian
from wand.api import library

suite_name, task_json, expected_mujoco = sys.argv[1:]
assert torch.cuda.is_available(), 'A100 GPU 런타임을 선택하세요.'
gpu = torch.cuda.get_device_name(0)
assert 'A100' in gpu, gpu
native = mujoco.mj_versionString()
assert (mujoco.__version__, native) == (expected_mujoco, expected_mujoco), (mujoco.__version__, native)
assert hasattr(library, 'MagickMotionBlurImage')
suite = benchmark.get_benchmark_dict()[suite_name]()
task_ids = json.loads(task_json)
assert suite.n_tasks == 2519 and all(0 <= task_id < suite.n_tasks for task_id in task_ids)
env = OffScreenRenderEnv(
    bddl_file_name=suite.get_task_bddl_file_path(task_ids[0]),
    camera_heights=256, camera_widths=256,
)
try:
    obs = env.reset()
    assert {'agentview_image', 'robot0_eye_in_hand_image'} <= obs.keys()
finally:
    env.close()
with tempfile.TemporaryDirectory() as temp:
    video = Path(temp) / 'test.mp4'
    with imageio.get_writer(str(video), fps=30) as writer:
        writer.append_data(np.zeros((32, 32, 3), dtype=np.uint8))
    assert video.read_bytes()[4:8] == b'ftyp'
    with imageio.get_reader(str(video)) as reader:
        assert reader.get_data(0).shape == (32, 32, 3)
software = {name: metadata.version(name) for name in (
    'torch', 'numpy', 'mujoco', 'robosuite', 'transformers', 'dlimp', 'imageio-ffmpeg',
)}
software['mujoco_native'] = native
software['python'] = sys.version.split()[0]
for name in ('transformers', 'dlimp'):
    direct_url = metadata.distribution(name).read_text('direct_url.json')
    if direct_url:
        software[name + '_commit'] = json.loads(direct_url).get('vcs_info', {}).get('commit_id')
print('VLADE_PREFLIGHT=' + json.dumps({'gpu': gpu, 'software': software}))
'''
probe = subprocess.run(
    [str(PYTHON), '-c', PREFLIGHT, SUITE, json.dumps(TASK_IDS), MUJOCO_VERSION],
    cwd=LIBERO, env=ENV, input='N\n', text=True, capture_output=True, timeout=300,
)
if probe.returncode:
    raise RuntimeError(probe.stdout[-4000:] + '\n' + probe.stderr[-5000:])
RUNTIME = json.loads(next(
    line.split('=', 1)[1] for line in probe.stdout.splitlines() if line.startswith('VLADE_PREFLIGHT=')
))
print('시뮬레이터·양쪽 카메라·MP4 쓰기/읽기 검사 통과', RUNTIME)

T0_ASSET = read_json(PROJECT / 'configs/storage/hf_assets.lock.json')['assets']['ava_t0']
RESOLVE_T0 = '''
import sys
from vlade.storage.assets import asset_record, cache_path, resolve_input

value = sys.argv[1]
path = resolve_input(value)
expected = asset_record('ava_t0')
if path.name != expected['local_name'] or path.parent.name != expected['revision'] or path.parents[1].name != 'ava_t0':
    raise ValueError(f'Expected a verified ava_t0 cache, got {path}')
print('VLADE_T0=' + str(path))
'''
hf_token = optional_secret('HF_TOKEN') or os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Colab Secrets에서 HF_TOKEN 접근을 허용하세요.')
resolved = capture(PYTHON, '-c', RESOLVE_T0, T0_CHECKPOINT, env={**ENV, 'HF_TOKEN': hf_token})
T0_SOURCE = Path(next(
    line.split('=', 1)[1] for line in resolved.splitlines() if line.startswith('VLADE_T0=')
))

# 평가기가 모델 코드를 동기화할 수 있도록, 검증된 캐시에 별도 쓰기 경로를 둡니다.
T0_VIEW = Path(tempfile.mkdtemp(prefix='t0-view-', dir=WORK))
for source in T0_SOURCE.iterdir():
    if source.is_file():
        target = T0_VIEW / source.name
        if source.suffix in {'.json', '.py'}:
            shutil.copy2(source, target)
        else:
            target.symlink_to(source)
for component in ('action_head', 'proprio_projector', 'vision_attn_weight_generator'):
    if len(list(T0_VIEW.glob(f'{component}*.pt'))) != 1:
        raise RuntimeError(f'T0 구성요소의 파일을 하나로 특정할 수 없습니다. {component}')
t0_stats_path = T0_VIEW / 'dataset_statistics.json'
t0_stats = read_json(t0_stats_path)
NORMALIZATION_KEY = T0_UNNORM_KEY or next(
    (key for key in (SUITE, f'{SUITE}_no_noops') if key in t0_stats), None
)
if not NORMALIZATION_KEY or not {'action', 'proprio'} <= t0_stats.get(NORMALIZATION_KEY, {}).keys():
    raise ValueError(f'사용할 수 있는 T0 정규화 키를 확인하세요. {list(t0_stats)}')
T0_STATS_SHA = hashlib.sha256(t0_stats_path.read_bytes()).hexdigest()
print('T0 HF 경로', f"{T0_ASSET['repo_id']}@{T0_ASSET['revision']}/{T0_ASSET['prefix']}/files")
print('T0 실제 정규화 키', NORMALIZATION_KEY)

In [ ]:
# @title 5. T0 rollout 15회 실행 및 paired 결과 저장
EVAL_RUNNER = r'''
import hashlib
import importlib.util
import inspect
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np


def initial_state_hash(initial_state):
    if initial_state is None:
        raise RuntimeError('Expected a fixed benchmark initial state.')
    state = np.ascontiguousarray(initial_state)
    digest = hashlib.sha256()
    digest.update(repr((state.shape, state.dtype.str)).encode())
    digest.update(state.tobytes())
    return digest.hexdigest()


def evaluate(script, output, task_ids, normalization_key, argv):
    sys.argv = [str(script), *argv]
    sys.path.insert(0, str(script.parent))
    spec = importlib.util.spec_from_file_location('vlade_context_eval', script)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    run = json.loads((output / 'run.json').read_text(encoding='utf-8'))
    suite = module.benchmark.get_benchmark_dict()[run['suite']]()
    if not all(0 <= task_id < suite.n_tasks for task_id in task_ids):
        raise ValueError('Task ID is outside the pinned benchmark.')
    stats_path = Path(run['checkpoint_view']) / 'dataset_statistics.json'
    if hashlib.sha256(stats_path.read_bytes()).hexdigest() != run['normalization']['stats_sha256']:
        raise RuntimeError('T0 normalization statistics changed after preflight.')
    source_stats = json.loads(stats_path.read_text(encoding='utf-8'))
    run_task = module.run_task
    run_episode = module.run_episode
    get_action = module.get_action
    log_message = module.log_message
    save_video = module.save_rollout_video
    task_signature = inspect.signature(run_task)
    episode_signature = inspect.signature(run_episode)
    tqdm = module.tqdm.tqdm
    records = []
    last_episode = None
    video_path = None

    def selected_tqdm(iterable, *args, **kwargs):
        # The pinned evaluator has one outer task loop and one per-task trial loop.
        if isinstance(iterable, range) and iterable == range(suite.n_tasks):
            iterable = task_ids
        return tqdm(iterable, *args, **kwargs)

    def fail_on_episode_error(message, *args, **kwargs):
        log_message(message, *args, **kwargs)
        if str(message).startswith('Episode error:'):
            raise RuntimeError(message)

    def use_t0_stats(cfg, model):
        if model.norm_stats.get(normalization_key) != source_stats[normalization_key]:
            raise RuntimeError(f'Model normalization statistics differ from T0: {normalization_key}')
        cfg.unnorm_key = normalization_key
        print(f'T0 normalization: {cfg.unnorm_key}', flush=True)

    def traced_episode(*args, **kwargs):
        nonlocal last_episode
        bound = episode_signature.bind(*args, **kwargs)
        bound.apply_defaults()
        digest = initial_state_hash(bound.arguments['initial_state'])
        query_count = 0
        reset_queries = []

        def traced_get_action(*action_args, **action_kwargs):
            nonlocal query_count
            query_count += 1
            if action_kwargs['temporal_context'] is None:
                reset_queries.append(query_count)
            return get_action(*action_args, **action_kwargs)

        module.get_action = traced_get_action
        try:
            result = run_episode(*args, **kwargs)
        finally:
            module.get_action = get_action
        if reset_queries != [1]:
            raise RuntimeError(f'Expected context reset at query 1, got {reset_queries}')
        last_episode = {'initial_state_sha256': digest, 'policy_queries': query_count,
                        'context_reset_queries': reset_queries}
        return result

    def traced_video(*args, **kwargs):
        nonlocal video_path
        path = Path(save_video(*args, **kwargs))
        if not path.is_relative_to(output) or not path.is_file() or not path.stat().st_size:
            raise RuntimeError(f'Expected an MP4 in the T0 run folder, got {path}')
        video_path = str(path)
        return str(path)

    def record_task(*args, **kwargs):
        nonlocal last_episode, video_path
        inputs = task_signature.bind(*args, **kwargs).arguments
        task_id = int(inputs['task_id'])
        expected = task_ids[len(records)] if len(records) < len(task_ids) else None
        if task_id != expected:
            raise RuntimeError(f'Expected task {expected}, got {task_id}')
        last_episode, video_path = None, None
        before = inputs['total_episodes'], inputs['total_successes']
        after = run_task(*args, **kwargs)
        success = after[1] - before[1]
        if after[0] - before[0] != 1 or success not in (0, 1) or not last_episode or not video_path:
            raise RuntimeError(f'Expected one completed rollout and MP4 for task {task_id}')
        records.append({'task_id': task_id, 'success': bool(success),
                        'video_path': video_path, **last_episode})
        temporary = output / 'task_results.tmp'
        temporary.write_text(json.dumps(records, indent=2) + '\n', encoding='utf-8')
        temporary.replace(output / 'task_results.json')
        print(f'Task {task_id}: success={bool(success)}, video={video_path}', flush=True)
        return after

    cfg = module.GenerateConfig()
    expected_defaults = {'initial_states_path': 'DEFAULT', 'num_steps_wait': 10,
                         'env_img_res': 256, 'center_crop': True, 'log_camera_wrist': True}
    default_mismatch = {key: {'k4': value, 't0': getattr(cfg, key)}
                        for key, value in expected_defaults.items() if getattr(cfg, key) != value}
    if default_mismatch:
        raise RuntimeError(f'Evaluator defaults differ: {default_mismatch}')
    module.tqdm = SimpleNamespace(tqdm=selected_tqdm)
    module.log_message = fail_on_episode_error
    module.check_unnorm_key = use_t0_stats
    module.run_episode = traced_episode
    module.save_rollout_video = traced_video
    module.run_task = record_task
    module.eval_libero()
    if [row['task_id'] for row in records] != task_ids:
        raise RuntimeError('Incomplete or reordered rollout results.')


if __name__ == '__main__':
    script, folder, tasks, norm_key, *argv = sys.argv[1:]
    evaluate(Path(script), Path(folder), json.loads(tasks), norm_key, argv)
'''


def run_t0():
    global RUN_DIR
    run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    RUN_DIR = OUTPUT_ROOT / run_id
    args = [item for pair in COMMON_ARGS.items() for item in pair]
    args += ['--pretrained_checkpoint', str(T0_VIEW), '--local_log_dir', str(RUN_DIR)]
    actual_args = argument_map(args)
    actual_shared = {key: value for key, value in actual_args.items() if key not in RUN_PATH_ARGS}
    shared = differences(COMMON_ARGS, actual_shared)
    require_match('T0와 K4 공통 평가 설정', COMMON_ARGS, actual_shared)
    model_differences = {
        'checkpoint': {'k4': K4_RUN['checkpoint'], 't0': str(T0_SOURCE)},
        'local_log_dir': {'k4': str(K4_DIR), 't0': str(RUN_DIR)},
        'normalization': {'k4': 'libero_10 ← libero_plus_mixdata', 't0': NORMALIZATION_KEY},
    }
    print('모델·출력 설정 차이', json.dumps(model_differences, ensure_ascii=False, indent=2))
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    metadata = {
        'status': 'running', 'model': 'T0', 'condition': 'episode_only',
        'reset_every_policy_queries': None, 'suite': SUITE, 'task_ids': TASK_IDS,
        'initial_states_path': 'DEFAULT', 'camera_observation_keys': ['agentview_image', 'robot0_eye_in_hand_image'],
        'evaluator_defaults': {'num_steps_wait': 10, 'env_img_res': 256, 'center_crop': True, 'log_camera_wrist': True},
        'initial_state_hash_format': 'sha256(repr((shape, dtype.str)).encode() + contiguous_state.tobytes())',
        'baseline_run_dir': str(K4_DIR), 'args': args,
        'vlade_commit': ACTUAL_PROJECT, 'ava_commit': ACTUAL_AVA,
        'evaluator_sha256': EVALUATOR_IDENTITY['sha256'],
        'k4_evaluator_sha256': K4_RUN['evaluator_sha256'],
        'evaluator_sha256_match': EVALUATOR_IDENTITY['match'],
        'libero_plus_revision': ACTUAL_LIBERO,
        'assets_revision': ACTUAL_ASSETS, 'runtime': RUNTIME,
        'checkpoint': {'local_path': str(T0_SOURCE), 'repo_id': T0_ASSET['repo_id'],
                       'revision': T0_ASSET['revision'], 'prefix': T0_ASSET['prefix']},
        'checkpoint_view': str(T0_VIEW),
        'normalization': {'key': NORMALIZATION_KEY, 'stats_sha256': T0_STATS_SHA},
        'shared_setting_differences': shared, 'model_setting_differences': model_differences,
    }
    write_json(RUN_DIR / 'run.json', metadata)
    write_json(RUN_DIR / 'task_results.json', [])
    log_path = RUN_DIR / 'console.log'
    print('T0 실행 폴더', RUN_DIR, flush=True)
    command = [str(PYTHON), '-u', '-c', EVAL_RUNNER, str(SCRIPT), str(RUN_DIR),
               json.dumps(TASK_IDS), NORMALIZATION_KEY, *args]
    try:
        with log_path.open('w', encoding='utf-8') as log:
            result = subprocess.run(command, cwd=AVA, env=ENV, stdout=log, stderr=subprocess.STDOUT)
        metadata['returncode'] = result.returncode
        if result.returncode:
            raise RuntimeError(f'T0 평가 실패 · {log_path}')
        metadata['status'] = 'complete'
    except BaseException:
        metadata['status'] = 'failed'
        raise
    finally:
        write_json(RUN_DIR / 'run.json', metadata)
        save_comparison(RUN_DIR)


run_t0()

In [ ]:
# @title 6. 저장된 비교 결과 다시 보기
# 새 런타임에서는 1~2번을 먼저 실행한 뒤 이전 실행의 폴더를 지정하세요.
RESULT_DIR = globals().get('RUN_DIR')  # 예: OUTPUT_ROOT / '<실행 타임스탬프>'
if RESULT_DIR is None:
    raise ValueError('5번을 실행하거나 RESULT_DIR에 기존 T0 실행 폴더를 지정하세요.')
save_comparison(Path(RESULT_DIR))